In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset, random_split
from PIL import Image
import pandas as pd
from tqdm import tqdm

In [ ]:
TRAIN_DIR = "/kaggle/input/Our_DataSources/train"  # We need to specify the path to the training data  
TEST_DIR = "/kaggle/input/Our_DataSources/test"    # We need to specify the path to the test data

NUM_CLASSES = 38   # Here we need figure it out with hown many for our job 
BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [ ]:
from torchvision.datasets import ImageFolder

dataset = ImageFolder(TRAIN_DIR, transform=train_transform)

train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size

train_data, val_data = random_split(dataset, [train_size, val_size])

val_data.dataset.transform = val_transform

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
class TestDataset(Dataset):
    def __init__(self, folder, transform=None):
        self.folder = folder
        self.transform = transform
        self.images = os.listdir(folder)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.folder, img_name)

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, os.path.splitext(img_name)[0]

test_dataset = TestDataset(TEST_DIR, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = models.efficientnet_b0(pretrained=True)

for param in model.parameters():
    param.requires_grad = True

model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.Adam(model.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

scaler = torch.cuda.amp.GradScaler()

In [ ]:
best_acc = 0

for epoch in range(EPOCHS):
    model.train()
    train_correct = 0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()

    train_acc = train_correct / len(train_loader.dataset)

    # VALIDATION
    model.eval()
    val_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            val_correct += (preds == labels).sum().item()

    val_acc = val_correct / len(val_loader.dataset)

    print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")

    scheduler.step()

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

In [ ]:
tta_transforms = [
    transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ]),
    transforms.Compose([
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
]

In [ ]:
predictions = []

for transform in tta_transforms:
    test_dataset.transform = transform

    temp_preds = []

    with torch.no_grad():
        for images, ids in tqdm(test_loader):
            images = images.to(device)
            outputs = model(images)

            probs = torch.softmax(outputs, dim=1)
            temp_preds.append(probs.cpu())

    if len(predictions) == 0:
        predictions = temp_preds
    else:
        predictions = [p + t for p, t in zip(predictions, temp_preds)]

final_preds = []

for batch in predictions:
    final_preds.extend(torch.argmax(batch, dim=1).numpy())

In [ ]:
## We need to configure this block accordingly.
image_ids = []

for _, ids in test_loader:
    image_ids.extend(ids)

submission = pd.DataFrame({
    "Image_ID": image_ids,
    "Label": final_preds
})

submission.to_csv("submission.csv", index=False)
print("Submission created!")